In [0]:
# Set shuffle partition to avoid conflict
spark.conf.set("spark.sql.shuffle.partitions", 200)

In [0]:
df = spark.read.format("csv").option("header",True).\
load("/Volumes/external-catlog/default/test-external-voume/Employee_Attrition.csv")
display(df)

In [0]:
# Transformations on dataframe and write data into delta table

from pyspark.sql.functions import col

high_risk_df = df.filter(
    (col("Attrition") == "No") & (col("JobSatisfaction").cast("int") < 3)
)

selected_columns = [
    "EmployeeNumber", "EmployeeName", "Department", "JobRole", "JobSatisfaction",
    "Age", "Gender", "MaritalStatus", "MonthlyIncome", "OverTime", "YearsAtCompany"
]
high_risk_selected_df = high_risk_df.select(*[c for c in selected_columns if c in high_risk_df.columns])

high_risk_selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catlog`.default.high_risk_attrition_employees")

In [0]:
display(spark.sql("DESCRIBE HISTORY `external-catlog`.default.high_risk_attrition_employees"))

In [0]:
df_delta = spark.read.format("delta").table("`external-catlog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
# Single record insert into delta table

from pyspark.sql import Row

dummy_data = [Row(
    EmployeeNumber="999999",
    Department="Dummy Dept",
    JobRole="Dummy Role",
    JobSatisfaction="1",
    Age="30",
    Gender="Other",
    MaritalStatus="Single",
    MonthlyIncome="0",
    OverTime="No",
    YearsAtCompany="0"
)]

dummy_df = spark.createDataFrame(dummy_data)

dummy_df.write.format("delta").mode("append").saveAsTable("`external-catlog`.default.high_risk_attrition_employees")

In [0]:
from delta.tables import DeltaTable


history_df = spark.sql("DESCRIBE HISTORY `external-catlog`.default.high_risk_attrition_employees")
display(history_df.select("version", "timestamp", "operation"))

In [0]:
df_delta = spark.read.format("delta").table("`external-catlog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
df_delta_v = spark.read.format("delta").option("versionAsOf", 0).table("`external-catlog`.default.high_risk_attrition_employees")
display(df_delta_v)

In [0]:
from pyspark.sql.functions import lit

timestamp = "2025-11-15T10:06:52.343+00:00"  # Replace with your desired timestamp

df_delta_ts = spark.read.format("delta").option("timestampAsOf", timestamp).table("`external-catlog`.default.high_risk_attrition_employees")
display(df_delta_ts)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catlog`.default.employee_transformed_data")

In [0]:
# Example transformation: filter employees with JobSatisfaction < 3
from pyspark.sql.functions import col

transformed_df = df.filter(col("JobSatisfaction").cast("int") < 3)

output_path = "/Volumes/external-catlog/default/employee_transformed_data/"

transformed_df.write.partitionBy("Department").format("parquet").mode("overwrite").save(output_path)